# Pitch Control Model Base

## Data Needed

**Player Tracking:** one row per player per frame
frame_id, time_s, player_id, team, x, y, vx, vy
**Ball Tracking:** one row per frame
frame_id, time_s, x, y, vx, vy

Likely will need to calulcate velocity because we are likely to only have player positions. The below code can calculate velocity. 

In [ ]:
df['vx'] = df.groupby('player_id')['x'].transform(
    lambda s: savgol_filter(s.diff() / dt, window_length=5, polyorder=2)
)

Must figure out how to do coordinate system properly. 
* Should team with ball always move left to right?
* Should team A always be going left to right and team B right to left?

The model runs in a frame by frame way. Loop through all the frames we have and do each one separately:
1. select the frame and pull the players and ball
2. build the player and ball objects using the classes
3. run the model on the single frame snapshot
4. repeat for all frames

## Optional Event Data for Pass Probability
**Events Table:**
frame_id, event_type, target_x, target_y, outcome
target values are the intended destination of the pass

In [1]:
"""
spearman_pitch_control.py
=========================

Faithful, numerically stable implementation of the dynamic pitch control model
described in:

    Spearman, W. (2018) "Beyond Expected Goals"

This version evaluates the Potential Pitch Control Field (PPCF) through 
direct numerical Euler integration of the control rate differential equation 
(Equation 3) over time.
"""

import numpy as np
from dataclasses import dataclass


# =============================================================================
# SECTION 0 — DATA STRUCTURES
# =============================================================================

@dataclass
class Player:
    id: str
    team: str
    x: float
    y: float
    vx: float = 0.0
    vy: float = 0.0


@dataclass
class Ball:
    x: float
    y: float
    vx: float = 0.0
    vy: float = 0.0
    player_id: str = None


# =============================================================================
# SECTION 1 — PITCH GRID
# =============================================================================

def build_pitch_grid(
    pitch_length=105.0,
    pitch_width=68.0,
    n_cols=105,
    n_rows=68
):
    """Create 2D pitch coordinate grid."""
    xs = np.linspace(0, pitch_length, n_cols)
    ys = np.linspace(0, pitch_width, n_rows)
    return np.meshgrid(xs, ys)


# =============================================================================
# SECTION 2 — PLAYER TIME TO INTERCEPT (TTI)
# =============================================================================

def player_time_to_intercept(
    player,
    target_x,
    target_y,
    vmax=5.0,
    acceleration=7.0
):
    """
    Spearman-style time-to-intercept calculation based on 'Beyond Expected Goals' (2018).
    
    Models a player accelerating at a constant rate (default 7.0 m/s^2) up to a 
    maximum speed (default 5.0 m/s) towards the target location from their current 
    position and velocity vector.
    """
    # Current player state
    p0 = np.array([player.x, player.y])
    v0_vec = np.array([player.vx, player.vy])

    # Standardize input targets to support meshgrids/arrays
    if isinstance(target_x, np.ndarray):
        targets = np.stack([target_x.ravel(), target_y.ravel()], axis=1)
    else:
        targets = np.array([[target_x, target_y]])

    # Calculate displacement vectors and absolute distances to targets
    displacement = targets - p0
    distances = np.linalg.norm(displacement, axis=1)
    
    # Prevent division by zero if the player is exactly on the target coordinates
    distances = np.where(distances == 0, 1e-8, distances)

    # Unit vectors pointing from the player to the targets
    unit_vectors = displacement / distances[:, np.newaxis]

    # Project the player's initial velocity vector onto the target direction
    v0_proj = np.sum(v0_vec * unit_vectors, axis=1)
    
    # Cap the initial projected velocity at vmax (in case player exceeds it)
    v0_proj = np.minimum(v0_proj, vmax)

    # Time required to reach maximum speed
    t_acc = (vmax - v0_proj) / acceleration

    # Distance covered during the acceleration phase
    d_acc = (v0_proj * t_acc) + (0.5 * acceleration * (t_acc ** 2))

    # Initialize time-to-intercept array
    tti = np.zeros_like(distances)

    # Case 1: Target is reached BEFORE completing the acceleration phase (d_acc >= distance)
    acc_phase_mask = d_acc >= distances
    # Solve quadratic kinematic equation: 0.5 * a * t^2 + v0 * t - distance = 0
    sqrt_term = np.maximum(v0_proj ** 2 + 2 * acceleration * distances, 0)
    tti[acc_phase_mask] = (-v0_proj[acc_phase_mask] + np.sqrt(sqrt_term[acc_phase_mask])) / acceleration

    # Case 2: Target is reached AFTER reaching maximum speed (d_acc < distance)
    cruise_phase_mask = ~acc_phase_mask
    d_cruise = distances[cruise_phase_mask] - d_acc[cruise_phase_mask]
    tti[cruise_phase_mask] = t_acc[cruise_phase_mask] + (d_cruise / vmax)

    # Format the return type to match input structure
    if isinstance(target_x, np.ndarray):
        return tti.reshape(target_x.shape)

    return float(tti[0])


# =============================================================================
# SECTION 3 — BALL TRAVEL TIME
# =============================================================================

def estimate_ball_flight_bounds(distances):
    """
    Simulates or approximates the minimum and maximum ball flight times 
    for a given distance based on aerodynamic drag profiles as described in Section 3.1.2.
    """
    # Min time: hard, low-driven ground pass (initial speed ~30 m/s, slowing down due to drag)
    # Max time: high-lofted air pass (spends significant time traveling through the air)
    
    # Ground pass / driven pass approximation with drag deceleration
    min_time = 0.04 * distances + 0.0003 * (distances ** 2)
    
    # Lofted / high pass approximation (takes longer to drop down)
    max_time = 0.12 * distances + 0.0015 * (distances ** 2)
    
    # Ensure minimum time is bounded below at 0
    return np.maximum(min_time, 0.0), np.maximum(max_time, 0.0)


def ball_travel_time(ball, target_x, target_y, attacking_players, player_tti_func):
    """
    Spearman-accurate ball travel time selection.
    
    Calculates the physical minimum and maximum time limits for the ball to reach a target,
    then selects the time of flight that most closely matches the arrival time of the 
    nearest attacking player.
    """
    # Compute straight-line distance from ball current position to target coordinates
    dx = target_x - ball.x
    dy = target_y - ball.y
    distances = np.sqrt(dx**2 + dy**2)
    
    # 1. Get physical boundaries of ball flight using aerodynamic drag approximations
    min_ball_time, max_ball_time = estimate_ball_flight_bounds(distances)
    
    # 2. Find the arrival time (Time-To-Intercept) for all attacking players to this spot
    # Supports both scalar targets and meshgrids
    if isinstance(target_x, np.ndarray):
        tti_shape = target_x.shape
        # Initialize an array to track the minimum TTI across all attackers at each grid cell
        nearest_attacker_tti = np.full(tti_shape, np.inf)
        for player in attacking_players:
            # Skip the player currently on the ball
            if player.id == ball.player_id:
                continue
            player_tti = player_tti_func(player, target_x, target_y)
            nearest_attacker_tti = np.minimum(nearest_attacker_tti, player_tti)
    else:
        # Scalar coordinate execution
        attacker_ttis = [
            player_tti_func(p, target_x, target_y) 
            for p in attacking_players if p.id != ball.player_id
        ]
        nearest_attacker_tti = min(attacker_ttis) if attacker_ttis else 0.0

    # 3. Selection Rule: Match the nearest attacking player's arrival time, 
    # bounded tightly by the physical minimum and maximum capabilities of the ball.
    selected_ball_time = np.clip(nearest_attacker_tti, min_ball_time, max_ball_time)
    
    if isinstance(target_x, np.ndarray):
        return selected_ball_time
    return float(selected_ball_time)


def ball_travel_time_grid(ball, grid_x, grid_y, attacking_players, player_tti_func):
    """ Vectorized grid wrapper for computing the PPCF meshgrid ball arrival matrices. """
    return ball_travel_time(ball, grid_x, grid_y, attacking_players, player_tti_func)


# =============================================================================
# SECTION 4 — ARRIVAL PROBABILITY & CONTROL RATE MODEL
# =============================================================================

def arrival_probability(t, tti, tti_sigma=0.54):
    """
    Spearman logistic arrival probability: P(T <= t)
    Represents f_j(t, r, T|s) from Equation 4.
    
    Note: default tti_sigma (s) updated to 0.54 seconds to match 
    the MAP estimate in Table 1.
    """
    exponent = -np.pi / np.sqrt(3.0) / tti_sigma * (t - tti)
    # Clip exponent safely to prevent exp overflow
    exponent = np.clip(exponent, -50, 50)
    return 1.0 / (1.0 + np.exp(exponent))


def control_rate(t, tti, lambda_j=3.99, tti_sigma=0.54):
    """
    Instantaneous rate of control for a player at time t.
    Based strictly on Equation 3: rate = f_j(T) * lambda_j
    
    Note: default lambda_j updated to 3.99 Hz to match the MAP 
    estimate for attackers in Table 1. (For defenders, this should 
    be multiplied by the parameter kappa = 1.72)
    """
    # Get the CDF from Equation 4
    P = arrival_probability(t, tti, tti_sigma)
    
    # The instantaneous control multiplier from Equation 3
    return P * lambda_j


# =============================================================================
# SECTION 5 — DYNAMIC PITCH CONTROL INTEGRATION
# =============================================================================

def dynamic_pitch_control(
    attacking_players,
    defending_players,
    ball,
    grid_x,
    grid_y,
    dt=0.04,
    max_time=10.0,
    vmax=5.0,
    acceleration=7.0,
    lambda_att=3.99,
    kappa=1.72,
    tti_sigma=0.54
):
    """
    Strict implementation of Spearman's Potential Pitch Control Field (PPCF).
    Numerically integrates Equation 3 over time.
    
    Parameters updated to Table 1 MAP estimates:
    - lambda_att: Attacker control rate (3.99 Hz)
    - kappa: Defender advantage multiplier (1.72)
    - tti_sigma: Uncertainty in arrival time (0.54 s)
    - acceleration / vmax: Player kinematic limits (7.0 m/s^2, 5.0 m/s)
    """
    shape = grid_x.shape
    n_cells = grid_x.size

    # Defender control rate is scaled by kappa (advantage for defending)
    lambda_def = lambda_att * kappa

    # 1. Pre-calculate Player Time-To-Intercept (TTI) surfaces using updated kinematic model
    TTI_att = np.stack([
        player_time_to_intercept(p, grid_x, grid_y, vmax, acceleration).ravel()
        for p in attacking_players
    ])
    TTI_def = np.stack([
        player_time_to_intercept(p, grid_x, grid_y, vmax, acceleration).ravel()
        for p in defending_players
    ])

    # 2. Ball Travel Time using the synchronization selection rule
    T_ball = ball_travel_time_grid(
        ball, grid_x, grid_y, attacking_players, player_time_to_intercept
    ).ravel()

    # Dynamic Allocation Arrays
    PPCF_att = np.zeros(n_cells)
    PPCF_def = np.zeros(n_cells)
    
    # Represents (1 - sum(PPCF_k)) from Equation 3: probability ball is uncontrolled
    PPCF_tot = np.zeros(n_cells)

    integration_times = np.arange(0.0, max_time, dt)

    for t in integration_times:
        # 3. Only evaluate cells where the ball has physically arrived
        active = t >= T_ball

        if not np.any(active):
            continue
            
        # Optimization: Stop calculating cells that are already >= 99.9% controlled
        calc_mask = active & (PPCF_tot < 0.999)
        if not np.any(calc_mask):
            continue

        # 4. Calculate control rates based strictly on Equation 3: rate = f_j(t) * lambda_j
        rate_att = np.sum([
            control_rate(t, TTI_att[i, calc_mask], lambda_att, tti_sigma)
            for i in range(len(attacking_players))
        ], axis=0)
        
        rate_def = np.sum([
            control_rate(t, TTI_def[i, calc_mask], lambda_def, tti_sigma)
            for i in range(len(defending_players))
        ], axis=0)
        
        # 5. Direct numerical integration of Equation 3
        # dPPCF = (1 - Total_PPCF) * rate * dt
        uncontrolled_prob = 1.0 - PPCF_tot[calc_mask]
        
        dPPCF_att = uncontrolled_prob * rate_att * dt
        dPPCF_def = uncontrolled_prob * rate_def * dt
        
        # 6. Accumulate control
        PPCF_att[calc_mask] += dPPCF_att
        PPCF_def[calc_mask] += dPPCF_def
        PPCF_tot[calc_mask] += dPPCF_att + dPPCF_def

    return PPCF_att.reshape(shape), PPCF_def.reshape(shape)


# =============================================================================
# SECTION 6 — PASS SUCCESS PROBABILITY
# =============================================================================

def pass_success_probability(
    attacking_players,
    defending_players,
    ball,
    target_x,
    target_y,
    dt=0.04,
    max_time=10.0,
    vmax=5.0,
    acceleration=7.0,
    lambda_att=3.99,
    kappa=1.72,
    tti_sigma=0.54
):
    """
    Calculates the probability of a successful pass to a specific target location.
    Based strictly on Section 3.2.1: Pass Probability is the attacking team's PPCF 
    at the destination.
    
    Optimized to compute only for the target coordinate rather than the whole pitch.
    """
    # Convert single target into a 1-element array format for vectorized functions
    tx = np.array([target_x])
    ty = np.array([target_y])

    # 1. Evaluate Player TTI only for the target location
    TTI_att = np.stack([
        player_time_to_intercept(p, tx, ty, vmax, acceleration)
        for p in attacking_players
    ])
    TTI_def = np.stack([
        player_time_to_intercept(p, tx, ty, vmax, acceleration)
        for p in defending_players
    ])

    # 2. Ball Travel Time to target
    T_ball = ball_travel_time(
        ball, tx, ty, attacking_players, player_time_to_intercept
    )

    # 3. Initialize integration variables
    PPCF_att = 0.0
    PPCF_def = 0.0
    PPCF_tot = 0.0
    
    lambda_def = lambda_att * kappa
    integration_times = np.arange(0.0, max_time, dt)

    # 4. Integrate exactly as before, but purely for the scalar target
    for t in integration_times:
        if t < T_ball:
            continue
            
        if PPCF_tot >= 0.999:
            break  # Stop early if the outcome is essentially decided

        # Calculate control rates based on Equation 3
        rate_att = np.sum([
            control_rate(t, TTI_att[i, 0], lambda_att, tti_sigma)
            for i in range(len(attacking_players))
        ])
        
        rate_def = np.sum([
            control_rate(t, TTI_def[i, 0], lambda_def, tti_sigma)
            for i in range(len(defending_players))
        ])
        
        # Euler integration
        uncontrolled_prob = 1.0 - PPCF_tot
        dPPCF_att = uncontrolled_prob * rate_att * dt
        dPPCF_def = uncontrolled_prob * rate_def * dt
        
        PPCF_att += dPPCF_att
        PPCF_def += dPPCF_def
        PPCF_tot += dPPCF_att + dPPCF_def

    # The returned float is exactly the attacking team's final pitch control at the target
    return float(PPCF_att)


# =============================================================================
# SECTION 7 — VERIFICATION RUN
# =============================================================================

if __name__ == "__main__":
    # 1. Build the tracking grid environment
    grid_x, grid_y = build_pitch_grid()

    # 2. Define a practical match scenario with explicit IDs and vectors
    attacking_players = [
        Player(id="A1", team="attack", x=52.0, y=34.0, vx=3.0, vy=0.0),
        Player(id="A2", team="attack", x=65.0, y=20.0, vx=1.0, vy=1.0)
    ]
    defending_players = [
        Player(id="D1", team="defense", x=60.0, y=34.0, vx=-2.0, vy=0.0),
        Player(id="D2", team="defense", x=70.0, y=50.0, vx=0.0, vy=-1.0)
    ]
    
    # Player "A1" starts on the ball
    ball = Ball(x=52.0, y=34.0, vx=5.0, vy=0.0, player_id="A1")

    # 3. Calculate full Pitch Control Fields across the meshgrid
    PPCF_att, PPCF_def = dynamic_pitch_control(
        attacking_players, defending_players, ball, grid_x, grid_y
    )

    print("Calculated Pitch Control Grids Successfully.")
    print(f"Max Attacking Pitch Control Found: {np.max(PPCF_att):.4f}")
    
    # Sample a specific index to verify convergence towards 1.0
    mid_y, mid_x = grid_x.shape[0] // 2, grid_x.shape[1] // 2
    total_control = PPCF_att[mid_y, mid_x] + PPCF_def[mid_y, mid_x]
    print(f"Total Grid Sum at center cell (Att + Def): {total_control:.4f}")

    # 4. Calculate Pass Success Probability using the point-optimized function
    # Notice: grid_x and grid_y are completely removed here to avoid redundant grid searches
    p_pass = pass_success_probability(
        attacking_players, 
        defending_players, 
        ball, 
        target_x=65.0, 
        target_y=28.0
    )
    
    print(f"Pass success probability at exact target (65, 34): {p_pass:.3f}")

Calculated Pitch Control Grids Successfully.
Max Attacking Pitch Control Found: 0.9987
Total Grid Sum at center cell (Att + Def): 0.9991
Pass success probability at exact target (65, 34): 0.502
